# 01 — Bronze Ingestion

Loads raw Google Trends CSVs and SEAB reference data into Bronze Delta tables.
This is the first task in the Databricks Workflow.

In [ ]:
%pip install pandas -q

In [ ]:
import pandas as pd
import time
from pyspark.sql import functions as F

In [ ]:
CATALOG = 'workspace'
SCHEMA  = 'default'

In [ ]:
# --- Segments: display name -> Google Trends keyword ---
SEGMENTS = {
    'Primary Math':        'primary math tuition singapore',
    'Primary English':     'primary english tuition singapore',
    'Primary Science':     'primary science tuition singapore',
    'Secondary Math':      'secondary math tuition singapore',
    'Secondary English':   'secondary english tuition singapore',
    'Secondary Biology':   'secondary biology tuition singapore',
    'Secondary Chemistry': 'secondary chemistry tuition singapore',
    'Secondary Physics':   'secondary physics tuition singapore',
    'JC Math':             'jc math tuition singapore',
    'JC Chemistry':        'jc chemistry tuition singapore',
    'JC Economics':        'jc economics tuition singapore',
}

SEG_ORDER = [
    'Primary Math','Primary English','Primary Science',
    'Secondary Math','Secondary English','Secondary Biology','Secondary Chemistry','Secondary Physics',
    'JC Math','JC Chemistry','JC Economics',
]

# --- MOE school term and holiday dates 2020-2026 ---
# Source: https://www.moe.gov.sg/news/press-releases
MOE_CALENDAR = {
    2020: {
        'term1': ('2020-01-02','2020-03-13'), 'term2': ('2020-03-23','2020-05-29'),
        'term3': ('2020-06-29','2020-09-04'), 'term4': ('2020-09-14','2020-11-20'),
        'march_hols': ('2020-03-14','2020-03-22'), 'june_hols': ('2020-05-30','2020-06-28'),
        'sept_hols': ('2020-09-05','2020-09-13'),  'year_end':  ('2020-11-21','2020-12-31'),
    },
    2021: {
        'term1': ('2021-01-04','2021-03-12'), 'term2': ('2021-03-22','2021-05-28'),
        'term3': ('2021-06-28','2021-09-03'), 'term4': ('2021-09-13','2021-11-19'),
        'march_hols': ('2021-03-13','2021-03-21'), 'june_hols': ('2021-05-29','2021-06-27'),
        'sept_hols': ('2021-09-04','2021-09-12'),  'year_end':  ('2021-11-20','2021-12-31'),
    },
    2022: {
        'term1': ('2022-01-03','2022-03-11'), 'term2': ('2022-03-21','2022-05-27'),
        'term3': ('2022-06-27','2022-09-02'), 'term4': ('2022-09-12','2022-11-18'),
        'march_hols': ('2022-03-12','2022-03-20'), 'june_hols': ('2022-05-28','2022-06-26'),
        'sept_hols': ('2022-09-03','2022-09-11'),  'year_end':  ('2022-11-19','2022-12-31'),
    },
    2023: {
        'term1': ('2023-01-03','2023-03-10'), 'term2': ('2023-03-20','2023-05-26'),
        'term3': ('2023-06-26','2023-09-01'), 'term4': ('2023-09-11','2023-11-17'),
        'march_hols': ('2023-03-11','2023-03-19'), 'june_hols': ('2023-05-27','2023-06-25'),
        'sept_hols': ('2023-09-02','2023-09-10'),  'year_end':  ('2023-11-18','2023-12-31'),
    },
    2024: {
        'term1': ('2024-01-02','2024-03-08'), 'term2': ('2024-03-18','2024-05-24'),
        'term3': ('2024-06-24','2024-08-30'), 'term4': ('2024-09-09','2024-11-15'),
        'march_hols': ('2024-03-09','2024-03-17'), 'june_hols': ('2024-05-25','2024-06-23'),
        'sept_hols': ('2024-08-31','2024-09-08'),  'year_end':  ('2024-11-16','2024-12-31'),
    },
    2025: {
        'term1': ('2025-01-02','2025-03-14'), 'term2': ('2025-03-24','2025-05-30'),
        'term3': ('2025-06-30','2025-09-05'), 'term4': ('2025-09-15','2025-11-21'),
        'march_hols': ('2025-03-15','2025-03-23'), 'june_hols': ('2025-05-31','2025-06-29'),
        'sept_hols': ('2025-09-06','2025-09-14'),  'year_end':  ('2025-11-22','2025-12-31'),
    },
    2026: {
        'term1': ('2026-01-02','2026-03-13'), 'term2': ('2026-03-23','2026-05-29'),
        'term3': ('2026-06-29','2026-09-04'), 'term4': ('2026-09-14','2026-11-20'),
        'march_hols': ('2026-03-14','2026-03-22'), 'june_hols': ('2026-05-30','2026-06-28'),
        'sept_hols': ('2026-09-05','2026-09-13'),  'year_end':  ('2026-11-21','2026-12-31'),
    },
}

# --- SEAB exam and results dates ---
# Source: https://www.seab.gov.sg/important-dates-for-candidates
SEAB_EVENTS = [
    {'event':'SA1_exam',       'start':'2020-04-27','end':'2020-05-08'},
    {'event':'SA1_exam',       'start':'2021-04-26','end':'2021-05-07'},
    {'event':'SA1_exam',       'start':'2022-04-25','end':'2022-05-06'},
    {'event':'SA1_exam',       'start':'2023-04-24','end':'2023-05-05'},
    {'event':'SA1_exam',       'start':'2024-04-22','end':'2024-05-03'},
    {'event':'SA1_exam',       'start':'2025-04-28','end':'2025-05-09'},
    {'event':'SA1_exam',       'start':'2026-04-27','end':'2026-05-08'},
    {'event':'SA2_exam',       'start':'2020-09-28','end':'2020-10-16'},
    {'event':'SA2_exam',       'start':'2021-09-27','end':'2021-10-15'},
    {'event':'SA2_exam',       'start':'2022-09-26','end':'2022-10-14'},
    {'event':'SA2_exam',       'start':'2023-09-25','end':'2023-10-13'},
    {'event':'SA2_exam',       'start':'2024-09-23','end':'2024-10-11'},
    {'event':'SA2_exam',       'start':'2025-09-29','end':'2025-10-17'},
    {'event':'SA2_exam',       'start':'2026-09-28','end':'2026-10-16'},
    {'event':'PSLE_exam',      'start':'2020-08-31','end':'2020-09-25'},
    {'event':'PSLE_exam',      'start':'2021-08-30','end':'2021-09-24'},
    {'event':'PSLE_exam',      'start':'2022-09-01','end':'2022-09-29'},
    {'event':'PSLE_exam',      'start':'2023-08-31','end':'2023-09-28'},
    {'event':'PSLE_exam',      'start':'2024-08-27','end':'2024-09-27'},
    {'event':'PSLE_exam',      'start':'2025-08-25','end':'2025-09-26'},
    {'event':'PSLE_exam',      'start':'2026-08-12','end':'2026-09-30'},
    {'event':'PSLE_results',   'start':'2020-11-25','end':'2020-11-25'},
    {'event':'PSLE_results',   'start':'2021-11-24','end':'2021-11-24'},
    {'event':'PSLE_results',   'start':'2022-11-23','end':'2022-11-23'},
    {'event':'PSLE_results',   'start':'2023-11-22','end':'2023-11-22'},
    {'event':'PSLE_results',   'start':'2024-11-27','end':'2024-11-27'},
    {'event':'PSLE_results',   'start':'2025-11-26','end':'2025-11-26'},
    {'event':'PSLE_results',   'start':'2026-11-24','end':'2026-11-25'},
    {'event':'OLevel_exam',    'start':'2020-10-05','end':'2020-11-06'},
    {'event':'OLevel_exam',    'start':'2021-10-04','end':'2021-11-05'},
    {'event':'OLevel_exam',    'start':'2022-10-03','end':'2022-11-04'},
    {'event':'OLevel_exam',    'start':'2023-10-02','end':'2023-11-03'},
    {'event':'OLevel_exam',    'start':'2024-10-07','end':'2024-11-08'},
    {'event':'OLevel_exam',    'start':'2025-10-06','end':'2025-11-07'},
    {'event':'OLevel_exam',    'start':'2026-10-05','end':'2026-11-06'},
    {'event':'OLevel_results', 'start':'2021-01-11','end':'2021-01-11'},
    {'event':'OLevel_results', 'start':'2022-01-12','end':'2022-01-12'},
    {'event':'OLevel_results', 'start':'2023-01-11','end':'2023-01-11'},
    {'event':'OLevel_results', 'start':'2024-01-10','end':'2024-01-10'},
    {'event':'OLevel_results', 'start':'2025-01-14','end':'2025-01-14'},
    {'event':'OLevel_results', 'start':'2026-01-13','end':'2026-01-15'},
    {'event':'ALevel_exam',    'start':'2020-10-08','end':'2020-11-20'},
    {'event':'ALevel_exam',    'start':'2021-10-07','end':'2021-11-19'},
    {'event':'ALevel_exam',    'start':'2022-10-06','end':'2022-11-18'},
    {'event':'ALevel_exam',    'start':'2023-10-05','end':'2023-11-17'},
    {'event':'ALevel_exam',    'start':'2024-10-10','end':'2024-11-22'},
    {'event':'ALevel_exam',    'start':'2025-10-09','end':'2025-11-21'},
    {'event':'ALevel_exam',    'start':'2026-10-08','end':'2026-11-27'},
    {'event':'ALevel_results', 'start':'2021-02-26','end':'2021-02-26'},
    {'event':'ALevel_results', 'start':'2022-02-25','end':'2022-02-25'},
    {'event':'ALevel_results', 'start':'2023-02-24','end':'2023-02-24'},
    {'event':'ALevel_results', 'start':'2024-02-22','end':'2024-02-22'},
    {'event':'ALevel_results', 'start':'2025-02-21','end':'2025-02-21'},
    {'event':'ALevel_results', 'start':'2026-02-19','end':'2026-02-23'},
]

df_events = pd.DataFrame(SEAB_EVENTS)
df_events['start'] = pd.to_datetime(df_events['start'])
df_events['end']   = pd.to_datetime(df_events['end'])
print(f'{len(SEGMENTS)} segments defined')
print(f'{len(df_events)} exam/results events loaded')

## Section 2: Google Trends Data — Load from CSV Files

Monthly search volume data downloaded from [trends.google.com](https://trends.google.com) (region: Singapore, 2020–present).
7 unique keywords are loaded and mapped to 11 segments — segments at different levels that share a subject
(e.g. Primary Math, Secondary Math, JC Math) use the same Trends signal but get different exam urgency regressors.

In [ ]:
BASE_PATH = '/Volumes/workspace/default/trends_data'

# Keyword -> actual filename on Databricks
KEYWORD_FILES = {
    'math tuition':      'time_series_SG_20200101-0000_20260511-1054.csv',
    'english tuition':   'time_series_SG_20200101-0000_20260511-1054-2.csv',
    'science tuition':   'time_series_SG_20200101-0000_20260511-1055.csv',
    'biology tuition':   'time_series_SG_20200101-0000_20260511-1055-2.csv',
    'chemistry tuition': 'time_series_SG_20200101-0000_20260511-1056.csv',
    'physics tuition':   'time_series_SG_20200101-0000_20260511-1056-2.csv',
    'economics tuition': 'time_series_SG_20200101-0000_20260511-1056-3.csv',
}

# Segment -> keyword (multiple segments share a keyword, differentiated by exam urgency regressors)
SEGMENT_KEYWORDS = {
    'Primary Math':        'math tuition',
    'Primary English':     'english tuition',
    'Primary Science':     'science tuition',
    'Secondary Math':      'math tuition',
    'Secondary English':   'english tuition',
    'Secondary Biology':   'biology tuition',
    'Secondary Chemistry': 'chemistry tuition',
    'Secondary Physics':   'physics tuition',
    'JC Math':             'math tuition',
    'JC Chemistry':        'chemistry tuition',
    'JC Economics':        'economics tuition',
}

# Load each unique keyword CSV once
keyword_data = {}
for keyword, filename in KEYWORD_FILES.items():
    path = f'{BASE_PATH}/{filename}'
    df_tmp = pd.read_csv(path, parse_dates=['Time'])
    df_tmp.columns = ['date', 'value']
    df_tmp['value'] = pd.to_numeric(df_tmp['value'], errors='coerce').fillna(0)
    # Filter to 2020 onwards
    df_tmp = df_tmp[df_tmp['date'] >= '2020-01-01']
    keyword_data[keyword] = df_tmp.set_index('date')['value']
    print(f'Loaded: {keyword} ({len(df_tmp)} monthly rows, {df_tmp.value.eq(0).mean()*100:.0f}% zeros)')

# Build wide DataFrame — one column per segment
frames = {seg: keyword_data[kw].rename(seg) for seg, kw in SEGMENT_KEYWORDS.items()}
df_trends_wide = pd.DataFrame(frames).reset_index()
df_trends_wide.columns.name = None
df_trends_wide['date'] = pd.to_datetime(df_trends_wide['date'])
seg_cols = [c for c in df_trends_wide.columns if c != 'date']

print(f'\nTrends table: {len(df_trends_wide)} monthly rows x {len(seg_cols)} segments')
display(spark.createDataFrame(df_trends_wide.head(5).astype(str)))

### Bronze Layer — Raw Ingestion

Land source data into Delta tables with no transformation.
Bronze is the audit trail — if anything breaks downstream, we reprocess from here.

In [ ]:
CATALOG = 'workspace'
SCHEMA  = 'default'

# Delta does not allow spaces in column names — replace with underscores for storage
df_trends_bronze = df_trends_wide.copy()
df_trends_bronze.columns = [c.replace(' ', '_') for c in df_trends_bronze.columns]

(spark.createDataFrame(df_trends_bronze.astype(str))
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_trends'))

# Raw SEAB exam and results events (reference data)
(spark.createDataFrame(pd.DataFrame(SEAB_EVENTS))
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_seab_events'))

print('Bronze tables written:')
print(f'  {CATALOG}.{SCHEMA}.bronze_trends        — {len(df_trends_bronze)} rows')
print(f'  {CATALOG}.{SCHEMA}.bronze_seab_events   — {len(SEAB_EVENTS)} rows')